In [1]:
import ast, json, re, random
from pathlib import Path
import pandas as pd

random.seed(42)

ZAYATS_PATH = "data/raw/zayats/switchboard_corrected_with_silver_reannotation.tsv"

FILLED_PAUSES = {"uh","um","er","erm","mm","hmm","huh","ah","eh"} # typical disfluency
SIL_TOKEN = "<sil>" # silent pause marker

TOKENS_COL = "ms_sentence"
TAGS_COL = "ms_disfl"

# Bare transcription artifacts (NOT real word fragments) -> dropped from BOTH sides.
NONLEXICAL = {"--", "-", "[silence]", "[noise]", "[laughter]", "[vocalized-noise]"}

# C_IE / C_IP sit INSIDE an outer reparandum in nested disfluencies, so for the fluent string we delete them too ?
REPARANDUM_TAGS = {"BE","IE","IP","BE_IP","C_IE","C_IP"}
KEEP_TAGS       = {"C","O"}
LIST_COLS = ["sentence","ms_sentence","comb_sentence","names","ms_names","comb_ann","tags","ms_disfl"]

In [2]:
def _maybe_eval(x):
    if isinstance(x, list): return x
    if isinstance(x, str) and x.strip().startswith("["):
        try: return ast.literal_eval(x)
        except Exception: return x
    return x

def load_zayats():
    df = pd.read_csv(ZAYATS_PATH, sep="\t")
    for c in LIST_COLS:
        if c in df.columns: df[c] = df[c].map(_maybe_eval)
    return df

df = load_zayats()
print("Columns:", list(df.columns))
df.head(50)

Columns: ['speaker', 'turn', 'sent_num', 'file', 'sentence', 'ms_sentence', 'comb_sentence', 'names', 'ms_names', 'comb_ann', 'first_name', 'ms_disfl']


,speaker,turn,sent_num,file,sentence,ms_sentence,comb_sentence,names,ms_names,comb_ann,first_name,ms_disfl
0,A,1,0,sw2005.trans,[okay],[okay],[okay],[uwA1_pw0],[uwA1_pw0],[O],uwA1_pw0,[O]
1,A,1,1,sw2005.trans,"[uh, first, um, i, need, to, know, uh, how, do...","[uh, first, um, i, need, to, know, uh, h-, how...","[uh, first, um, i, need, to, know, uh, h-, how...","[uwA1_pw1, uwA1_pw2, uwA1_pw3, uwA1_pw4, uwA1_...","[uwA1_pw1, uwA1_pw2, uwA1_pw3, uwA1_pw4, uwA1_...","[O, O, O, O, O, O, O, O, DEL, O, O, O, O, O, O...",uwA1_pw1,"[O, O, O, O, O, O, O, O, BE_IP, C, O, O, O, BE..."
2,B,2,0,sw2005.trans,"[well, of, course, it, 's, you, know, it, 's, ...","[well, of, course, it, 's, you, know, it, 's, ...","[well, of, course, it, 's, you, know, it, 's, ...","[uwB2_pw0, uwB2_pw1, uwB2_pw2, uwB2_pw3_a, uwB...","[uwB2_pw0, uwB2_pw1, uwB2_pw2, uwB2_pw3_a, uwB...","[O, O, O, O, O, O, O, O, O, DEL, O, O, O, O, O...",uwB2_pw0,"[O, O, O, BE, IP, C, C, C, C, O, O, O, O, O, O..."
3,A,3,0,sw2005.trans,[yes],[yes],[yes],[uwA3_pw28],[uwA3_pw28],[O],uwA3_pw28,[O]
4,A,3,1,sw2005.trans,[yeah],[yeah],[yeah],[uwA3_pw29],[uwA3_pw29],[O],uwA3_pw29,[O]
5,B,4,0,sw2005.trans,"[i, 'd, be, very, very, careful, and, uh, you,...","[i, 'd, be, very, very, careful, and, uh, you,...","[i, 'd, be, very, very, careful, and, uh, you,...","[uwB4_pw44_a, uwB4_pw44_b, uwB4_pw45, uwB4_pw4...","[uwB4_pw44_a, uwB4_pw44_b, uwB4_pw45, uwB4_pw4...","[O, O, O, O, O, O, O, O, O, O, O, O, O]",uwB4_pw44_a,"[O, O, O, O, O, O, BE_IP, O, O, O, O, O, O]"
6,B,4,1,sw2005.trans,"[uh, our]","[uh, our]","[uh, our]","[uwB4_pw56, uwB4_pw57]","[uwB4_pw56, uwB4_pw57]","[O, O]",uwB4_pw56,"[O, O]"
7,B,4,2,sw2005.trans,"[had, t-, place, my, mother, in, a, nursing, h...","[had, t-, place, my, mother, in, a, nursing, h...","[had, t-, place, my, mother, in, a, nursing, h...","[uwB4_pw58, uwB4_pw59, uwB4_pw60, uwB4_pw61, u...","[uwB4_pw58, uwB4_pw59, uwB4_pw60, uwB4_pw61, u...","[O, O, O, O, O, O, O, O, O]",uwB4_pw58,"[O, O, O, O, O, O, O, O, O]"
8,B,4,3,sw2005.trans,"[she, had, a, rather, massive, stroke, about, ...","[she, had, a, rather, massive, stroke, about, ...","[she, had, a, rather, massive, stroke, about, ...","[uwB4_pw67, uwB4_pw68, uwB4_pw69, uwB4_pw70, u...","[uwB4_pw67, uwB4_pw68, uwB4_pw69, uwB4_pw70, u...","[O, O, O, O, O, O, O, O, INS, O, O, O, DEL, O,...",uwB4_pw67,"[O, O, O, O, O, O, BE_IP, C_IP, O, BE_IP, C, O..."
9,A,5,0,sw2005.trans,[uh-huh],[um-hum],"[um-hum, uh-huh]",[uwA5_pw30],[uwA5_pw30],"[SUB_MS, SUB_TREE]",uwA5_pw30,[O]


In [3]:
bad_rows = []

for idx, row in df.iterrows():
    nt = len(row[TOKENS_COL])
    ng = len(row[TAGS_COL])
    diff = nt - ng

    if diff != 0:
        bad_rows.append({
            "idx": idx,
            "file": row.get("file"),
            "speaker": row.get("speaker"),
            "turn": row.get("turn"),
            "sent_num": row.get("sent_num"),
            "n_tokens": nt,
            "n_tags": ng,
            "diff": diff,
            "tokens": row[TOKENS_COL],
            "tags": row[TAGS_COL],
        })

bad_df = pd.DataFrame(bad_rows)

total = len(df)
bad = len(bad_df)
good = total - bad

print(f"Total rows:   {total:,}")
print(f"Aligned rows: {good:,} ({good / total * 100:.2f}%)")
print(f"Bad rows:     {bad:,} ({bad / total * 100:.2f}%)")

display(bad_df.head(20))

Total rows:   193,085
Aligned rows: 174,328 (90.29%)
Bad rows:     18,757 (9.71%)


,idx,file,speaker,turn,sent_num,n_tokens,n_tags,diff,tokens,tags
0,8,sw2005.trans,B,4,3,17,15,2,"[she, had, a, rather, massive, stroke, about, ...","[O, O, O, O, O, O, BE_IP, C_IP, O, BE_IP, C, O..."
1,21,sw2005.trans,B,10,0,14,13,1,"[we, were, not, really, happy, with, //, the, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O]"
2,25,sw2005.trans,B,12,2,18,17,1,"[but, it, 's, really, a, big, uh, big, //, dec...","[O, O, O, O, O, BE_IP, C, C, O, BE, IP, C, C, ..."
3,28,sw2005.trans,B,14,1,22,20,2,"[course, there, 's, you, know, there, 's, not,...","[O, BE, IP, C, C, C, C, O, O, O, O, O, O, O, O..."
4,41,sw2005.trans,A,23,1,9,7,2,"[and, um, that, was, even, harder, --, --, act...","[O, O, O, O, O, O, O]"
5,47,sw2005.trans,A,29,0,31,29,2,"[i, mean, she, tr-, she, had, she, had, childr...","[O, O, BE, IP, C_IE, C_IP, C, C, O, O, O, O, O..."
6,58,sw2005.trans,A,37,1,42,40,2,"[i, mean, for, somebody, who, is, you, know, f...","[O, O, O, O, O, BE_IP, O, O, O, O, O, O, O, BE..."
7,68,sw2005.trans,A,45,2,30,28,2,"[probably, one, of, the, biggest, decisions, i...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
8,76,sw2005.trans,A,51,0,27,26,1,"[and, and, just, the, children, were, involved...","[BE_IP, C, O, O, O, O, O, O, O, O, O, O, O, O,..."
9,79,sw2005.trans,A,53,0,8,6,2,"[and, everybody, had, duties, --, --, to, perf...","[O, O, O, O, O, O]"


In [4]:
from collections import Counter
import pandas as pd

tag_counts = Counter()

for tags in df[TAGS_COL]:
    tag_counts.update(tags)

tag_counts_df = pd.DataFrame(
    tag_counts.most_common(),
    columns=["tag", "count"]
)

total_tags = tag_counts_df["count"].sum()
tag_counts_df["pct"] = tag_counts_df["count"] / total_tags * 100

tag_counts_df

,tag,count,pct
0,O,1340106,87.648664
1,C,95284,6.231981
2,BE_IP,33443,2.187315
3,IP,18595,1.216193
4,BE,17259,1.128812
5,IE,11112,0.726772
6,C_IP,7431,0.486019
7,C_IE,5722,0.374243


In [5]:
def show_alignment(tokens, tags, width=14):
    for t, g in zip(tokens, tags):
        mark = ""
        if g in REPARANDUM_TAGS:        mark = "  <-- reparandum (deleted to fluentize)"
        elif t.lower() in FILLED_PAUSES: mark = "  <-- filled pause"
        elif t in NONLEXICAL:            mark = "  <-- transcription artifact (dropped both sides)"
        elif re.match(r"^[a-z]+[a-z']*-$", t.lower()): mark = "  <-- word fragment"
        print(f"{t:<{width}} {g:<6}{mark}")

# show the longest sentence available
i = max(range(len(df)), key=lambda k: len(df.iloc[k][TOKENS_COL]))
print(f"row {i}  (speaker={df.iloc[i].get('speaker','?')}, turn={df.iloc[i].get('turn','?')})\n")
show_alignment(df.iloc[i][TOKENS_COL], df.iloc[i][TAGS_COL])

row 175883  (speaker=B, turn=54)

um             O       <-- filled pause
i              BE      <-- reparandum (deleted to fluentize)
think          IP      <-- reparandum (deleted to fluentize)
i              C     
think          C     
one            O     
of             O     
the            O     
main           O     
things         O     
that           O     
's             O     
really         O     
important      O     
to             O     
spend          O     
money          O     
on             O     
is             O     
education      O     
not            O     
support        O     
or             BE_IP   <-- reparandum (deleted to fluentize)
or             C     
research       O     
per            O     
se             O     
but            BE_IP   <-- reparandum (deleted to fluentize)
but            C     
education      O     
because        O     
a              O     
lot            O     
of             O     
people         O     
sort           O     


In [6]:
TYPICAL  = {"filled_pause","silent_pause"}
ATYPICAL = {"reparandum","fragment"}

FRAG_RE = re.compile(r"^[a-z]+[a-z']*-$")   # real partial words: h-, bir-, th-  (not bare --/-)
def is_fragment(tok):
    return bool(FRAG_RE.match(tok.lower()))

def drop_nonlexical(tokens, tags):
    # strip transcription artifacts from tokens+tags in lockstep (keeps alignment)
    kept = [(t, g) for t, g in zip(tokens, tags) if t not in NONLEXICAL]
    if not kept: return [], []
    t, g = zip(*kept); return list(t), list(g)

def classify(tok, tag):
    # Disfluency class of a token, or None if fluent.
    if tag in REPARANDUM_TAGS:
        return "fragment" if is_fragment(tok) else "reparandum"
    if tok == SIL_TOKEN:        return "silent_pause"
    if tok.lower() in FILLED_PAUSES: return "filled_pause"
    if is_fragment(tok):        return "fragment"
    return None

def render(tokens, tags, allow):
    # Keep fluent tokens plus any disfluency classes in `allow`.
    out = []
    for tok, tag in zip(tokens, tags):
        cls = classify(tok, tag)
        if cls is None or cls in allow:
            out.append(tok)
    return out

_CLITICS = {"'s","'re","'ve","'ll","'d","'m","n't","'t"}
_MERGE = {"gon na":"gonna","wan na":"wanna","got ta":"gotta","gim me":"gimme","lem me":"lemme"}
def detok(tokens):
    # PTB tokens -> natural text:  that 's -> that's,  do n't -> don't,  they 're -> they're
    s = ""
    for j, tok in enumerate(tokens):
        if j == 0:
            s = tok
        elif tok.lower() in _CLITICS or tok in {",",".","?","!",";",":","%",")","]","}","''"} or tok.startswith("'"):
            s += tok
        else:
            s += " " + tok
    for a, b in _MERGE.items():
        s = s.replace(a, b)
    return s

def show_pair(tokens, tags):
    toks, tgs = drop_nonlexical(tokens, tags)          # same cleaning as make_pairs
    print("FLUENT     :", detok(render(toks, tgs, set())))
    print("DISFLUENT  :", detok(render(toks, tgs, TYPICAL|ATYPICAL)))
    print("  atypical :", detok(render(toks, tgs, ATYPICAL)))
    print("  typical  :", detok(render(toks, tgs, TYPICAL)))

show_pair(df.iloc[i][TOKENS_COL], df.iloc[i][TAGS_COL])

FLUENT     : i think one of the main things that's really important to spend money on is education not support or research per se but education because a lot of people sort of get drummed out of the work place and out of society you know kind of like lepers would you's really it's it's not necessary until until the disease reaches its final stage you know becomes really infectious and i think that would be the most effective support they is for other people to understand you know try to understand what they're going through and they're not you know they don't they do have to be cut off from
DISFLUENT  : um i think i think one of the main things that's really important to spend money on is education not support or or research per se but but education because a lot of people sort of get drummed out of the the work place and out of society you know kind of like lepers would you know and it's really it's it's really not necessary uh until until the uh disease reaches its final stage you kn

In [7]:
show_pair(df.iloc[8][TOKENS_COL], df.iloc[8][TAGS_COL])

FLUENT     : she had a rather massive stroke eight months ago
DISFLUENT  : she had a rather massive stroke about uh uh s- eight months ago
  atypical : she had a rather massive stroke about uh s- eight months ago
  typical  : she had a rather massive stroke uh eight months ago


In [8]:
def make_pairs(df):
    recs = []
    for _, r in df.iterrows():
        toks, tags = r[TOKENS_COL], r[TAGS_COL]
        if not (isinstance(toks, list) and isinstance(tags, list) and len(toks) == len(tags)):
            continue
        toks, tags = drop_nonlexical(toks, tags)      # remove --, [noise], etc. from both sides
        if not toks: continue
        labels = [classify(t, g) for t, g in zip(toks, tags)]
        fluent = render(toks, tags, set())
        rec = {
            "fluent":              detok(fluent),
            "disfluent":           detok(render(toks, tags, TYPICAL|ATYPICAL)),
            "disfluent_atypical":  detok(render(toks, tags, ATYPICAL)),
            "disfluent_typical":   detok(render(toks, tags, TYPICAL)),
            "n_tokens_fluent":     len(fluent),
            "n_tokens_disfluent":  len(toks),
            "n_disfl_tokens":      sum(1 for l in labels if l is not None),
        }
        rec["disfl_rate"] = rec["n_disfl_tokens"] / len(toks) if toks else 0.0
        # nested disfluencies are where silver-tag fluent reconstruction is least reliable
        rec["suspect"] = any(g in ("C_IE", "C_IP") for g in tags)
        for c in ("file","speaker","turn","sent_num"):   # keeping metadata
            if c in df.columns: rec[c] = r[c]
        recs.append(rec)
    return pd.DataFrame(recs)

pairs = make_pairs(df)
n_susp = int(pairs["suspect"].sum())
print(f"{len(pairs):,} usable pairs  ({n_susp:,} flagged 'suspect' = nested disfluency, review/drop)")
pairs[["fluent","disfluent"]].head(30)

173,475 usable pairs  (4,578 flagged 'suspect' = nested disfluency, review/drop)


,fluent,disfluent
0,okay,okay
1,first i need to know how do you feel about sen...,uh first um i need to know uh h- how do you fe...
2,well of course you know it's it one of the las...,well of course it's you know it's it one of th...
3,yes,yes
4,yeah,yeah
5,i'd be very very careful you know checking the...,i'd be very very careful and uh you know check...
6,our,uh our
7,had place my mother in a nursing home,had t- place my mother in a nursing home
8,um-hum,um-hum
9,and i was fortunate in that,and uh we were i was fortunate in that


In [9]:
def corpus_stats(p):
    return {
        "n_sentences":            len(p),
        "avg_tokens_fluent":      round(p["n_tokens_fluent"].mean(), 2),
        "avg_tokens_disfluent":   round(p["n_tokens_disfluent"].mean(), 2),
        "total_fluent_tokens":    int(p["n_tokens_fluent"].sum()),
        "total_disfluent_tokens": int(p["n_tokens_disfluent"].sum()),
        "disfluency_rate_%":      round(100 * p["n_disfl_tokens"].sum() / max(1, p["n_tokens_disfluent"].sum()), 1),
    }

pd.DataFrame([corpus_stats(pairs)])

,n_sentences,avg_tokens_fluent,avg_tokens_disfluent,total_fluent_tokens,total_disfluent_tokens,disfluency_rate_%
0,173475,6.17,6.75,1070835,1171593,8.6
